# Structured streaming Demo

## 1. Connexion à Kafka
Le connecteur correspond aux versions Spark et Scala du conteneur.

In [ ]:
# Imports cell

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

from pyspark.sql.functions import window, col, count

# Convert timestamp (Unix epoch in seconds) to timestamp type
from pyspark.sql.functions import from_unixtime



from pyspark.sql.functions import avg, sum as spark_sum, abs as spark_abs


In [ ]:
# Aligner le connecteur Kafka sur le Spark/Scala installé dans l'image fournie.
import pyspark
from pathlib import Path
from pyspark.find_spark_home import _find_spark_home

spark_version = pyspark.__version__
scala_jar = next((Path(_find_spark_home()) / "jars").glob("scala-library-*.jar"))
scala_version = ".".join(scala_jar.stem.removeprefix("scala-library-").split(".")[:2])
connector = f"org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}"
spark = (SparkSession.builder
    .appName("KafkaStreamingPrototype")
    .config("spark.jars.packages", connector)
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark :", spark.version, "Connecteur :", connector)
# Relancer toutes les cellules ne doit pas multiplier les requêtes actives.
for query in spark.streams.active:
    query.stop()


## 2. Lecture du stream
Kafka est joint par son adresse interne `kafka:19092`. `earliest` permet de relire les messages déjà produits.

In [ ]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


## 3. Parsing des événements Wikimedia
Le JSON est parsé avec un schéma explicite ; les analyses utilisent le timestamp de la modification.

In [ ]:
# Define a simplified schema - just the key fields for analysis
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", LongType(), True),
        StructField("new", LongType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), simple_wikimedia_schema).alias("data")
).select("data.*")

# Le timestamp Wikimedia est exprimé en secondes Unix : event time, en UTC.
edits_df = (parsed_df
    .filter(col("type") == "edit")
    .withColumn("event_time", from_unixtime(col("timestamp")).cast("timestamp"))
    .filter(col("event_time").isNotNull()))
edits_df.printSchema()


## 4. Démo : edits bots vs humains
Nombre de modifications par heure et selon `bot`. Le mode `complete` affiche toute la table dans les logs Docker.

In [ ]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = edits_df \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

In [ ]:

# Use console output (goes to Docker logs)
streaming_query_df = windowed_counts_df \
    .writeStream \
    .format("console") \
    .queryName("bots_par_heure") \
    .outputMode("complete") \
    .option("truncate", False) \
    .trigger(processingTime="10 seconds") \
    .start()

## 5. Exploration : taille des modifications
`length.new - length.old` donne la variation signée en octets : positive pour un ajout, négative pour une suppression. Les longueurs absentes sont exclues. On compare moyenne signée, moyenne absolue et total par valeur de `bot`.

In [ ]:
sized_edits_df = (edits_df
    .filter(col("length.old").isNotNull() & col("length.new").isNotNull())
    .withColumn("size_delta", col("length.new") - col("length.old")))
size_stats_df = sized_edits_df.groupBy("bot").agg(
    count("*").alias("edits_with_length"),
    avg("size_delta").alias("mean_delta_bytes"),
    avg(spark_abs(col("size_delta"))).alias("mean_absolute_delta_bytes"),
    spark_sum("size_delta").alias("total_delta_bytes"),
)
size_query = (size_stats_df.writeStream
    .queryName("taille_modifications").format("console")
    .outputMode("complete").option("truncate", False)
    .trigger(processingTime="10 seconds").start())


## 6. Tumbling Window
Fenêtres de 5 minutes sur l’event time. Le pas implicite vaut 5 minutes : les fenêtres ne se chevauchent pas. Le watermark de 2 minutes permet de gérer les événements tardifs et de libérer les anciennes fenêtres en mode `update`.

In [ ]:
timed_edits_df = edits_df.withWatermark("event_time", "2 minutes")
tumbling_counts_df = timed_edits_df.groupBy(
    window(col("event_time"), "5 minutes")
).agg(count("*").alias("edit_count"))
tumbling_query = (tumbling_counts_df.writeStream
    .queryName("tumbling_5_minutes").format("console")
    .outputMode("update").option("truncate", False)
    .trigger(processingTime="10 seconds").start())


## 7. Overlapping / Sliding Window
Fenêtres de 10 minutes, avec un slide de 5 minutes, sur l’event time. La durée dépasse le pas : une modification appartient à deux fenêtres. On compte les edits par fenêtre et selon `minor`. Le mode `update` affiche les lignes modifiées à chaque micro-batch.

In [ ]:
sliding_counts_df = timed_edits_df.groupBy(
    window(col("event_time"), "10 minutes", "5 minutes"), col("minor")
).agg(count("*").alias("edit_count"))
sliding_query = (sliding_counts_df.writeStream
    .queryName("sliding_10_minutes_pas_5").format("console")
    .outputMode("update").option("truncate", False)
    .trigger(processingTime="10 seconds").start())


## Observer et arrêter les requêtes
Les quatre requêtes démarrent sans bloquer les cellules suivantes. Observer leurs tables avec `docker logs -f pyspark_notebook` ; les sorties peuvent s’entrelacer. Après quelques micro-batches, réexécuter la cellule de statut : `exception` doit rester `None`. Pour arrêter, exécuter les deux lignes commentées.

In [ ]:
for query in spark.streams.active:
    print(query.name, "active:", query.isActive, "exception:", query.exception())
    print("Dernier micro-batch :", query.lastProgress)

# À exécuter uniquement À la fin de l'observation :
# for query in spark.streams.active:
#     query.stop()
